# Downsampling Algorithm for EIS data

There are a variety of statistical methods for downsampling timeseries data of various shapes, and yet the peculiar nature of electrochemical impedance spectroscopy data presented an opportunity for further developmment.

Window functions like simple moving averages or largest triangle three buckets are great for densely populated polynomials or otherwise uniformly populated datasets, but fail when considering large enough datasets with extreme minutia, like sparse asymptotes.

### Background and Descriptions

This algorithm targets galvanostatic test data for electrolysis stacks. The galvanostatic test reads data at a low interval, typically just a few seconds, for thousands of hours. The measurements taken are voltage, and current, among other values associated with EIS data.

#### Voltage data
Voltage data is characterized by several segments of data with a slope approximating infinity as a voltage is applied across the electrolysis stack. The performance of the stack degrades over time, showing a slightly negative trend, with many samples also taken at higher or lower voltages.

#### Current data
Current data is characterized by an initial ramp up, and many millions of readings at a constant current. The current changes based on the state of the system, for example whether the stack is heating up, or on load.

### Downsampling

The literature describe a few algoritms designed to downsample timeseries data while preserving their shape. First, the Ramer–Douglas–Peucker algorithm simplifies curves by decimating curves into representative line segments that result in a curve with fewer points. This algorithm however failed to account for the distinct asymptotes in EIS data.

Recent advances in downsampling algorithms for timeseries data from industrial processes have similarly developed a methodology for intelligent downsampling. The multivariate adaptive downsampling algorithm was replicated in Python and applied to sample EIS data, but because EIS data is so densely populated, it fails to interpret any distinct time domains where variance in the data exist. 

## Results
This algorithm uses the `scipy` `argrelextrema` function to calculate relative extrema in the data, neatly identifying variance like asymptotes in EIS data.

In [ ]:
##############################
# DOWNSAMPLING
##############################

import json
import os
import numpy as np
from scipy.signal import argrelextrema
import time as timer

# Target directory for downsampling
fixture_dir = "./FCE/stack1"

anonymized_time = np.array([])
anonymized_voltage = np.array([])
anonymized_current = np.array([])

downsampled_time = np.array([])
downsampled_current = np.array([])
downsampled_voltage = np.array([])

# Downsample to 10,000 points
num_files = len(os.listdir(fixture_dir))
n = 10000 // num_files

for filename in os.listdir(fixture_dir):

    start_time = timer.time()
    
    if filename.endswith(".json"):
        with open(os.path.join(fixture_dir, filename), 'r') as f:
            fixtures = json.load(f)

            time = []
            voltage = []
            current = []

            for obj in fixtures:
                fields = obj['fields']
                data = fields['data']
                time.append(data['time'])
                voltage.append(data['anonymized_voltage'])
                current.append(data['anonymized_current'])

            # Downsample
            time = np.array(time)
            anonymized_time = np.concatenate([anonymized_time, time])
            voltage = np.array(voltage)
            anonymized_voltage = np.concatenate([anonymized_voltage, voltage])
            current = np.array(current)
            anonymized_current = np.concatenate([anonymized_current, current])

            # Detect peaks and valleys in voltage data
            voltage_peaks = argrelextrema(voltage, np.greater, order=5)[0]
            voltage_valleys = argrelextrema(voltage, np.less, order=5)[0]
            voltage_critical = np.unique(np.concatenate([voltage_peaks, voltage_valleys]))

            # Detect asymptotes in current and voltage data
            voltage_diff = np.abs(np.diff(voltage))
            current_diff = np.abs(np.diff(current))

            voltage_asymptotes = np.where(voltage_diff > np.percentile(voltage_diff, 95))[0] + 1
            current_asymptotes = np.where(current_diff > np.percentile(current_diff, 95))[0] + 1

            # Combine critical indices
            voltage_critical = np.unique(np.concatenate([voltage_critical, voltage_asymptotes]))
            current_critical = current_asymptotes
            all_critical = np.concatenate([voltage_critical, current_critical, [0, len(time) - 1]])

            weights = np.zeros(len(time))
            weights[1:] += voltage_diff
            weights[1:] += current_diff

            if len(all_critical) > 0:
                sorted_indices = np.argsort(weights[all_critical])[-n:]
                keep_indices = all_critical[sorted_indices]
            else:
                keep_indices = np.linspace(0, len(time)-1, n, dtype=int)
            
            if len(keep_indices) < n and len(time) > n:
                keep_indices = np.linspace(0, len(time)-1, n, dtype=int)

            # Extract downsampled time
            downsampled_time = np.concatenate([downsampled_time, time[keep_indices]])
            downsampled_voltage = np.concatenate([downsampled_voltage, voltage[keep_indices]])
            downsampled_current = np.concatenate([downsampled_current, current[keep_indices]])

            end_time = timer.time()
            print(f"Processed {filename} in {end_time - start_time} seconds")


In [13]:
##############################
# VISUAL COMPARISONS
##############################
import matplotlib.pyplot as plt

In [ ]:
# Voltage Comparison
plt.figure(figsize=(12, 6))
plt.scatter(anonymized_time, anonymized_voltage, label='Original Voltage', color='lightblue', alpha=0.5)
plt.scatter(downsampled_time, downsampled_voltage, label="Olive Tree Voltage", color='blue', alpha=1, s=10)
plt.legend()
plt.show()

In [ ]:
# Current Comparison
plt.figure(figsize=(12, 6))
plt.scatter(anonymized_time, anonymized_current, label='Original Current', color='lightblue', alpha=0.5)
plt.scatter(downsampled_time, downsampled_current, label="EIS Downsample", color='blue', alpha=1, s=10)
plt.legend()
plt.show()